# Sustainability Aware Asset Management
## Group BB — Emerging Markets | Scope 1

---

**Matthieu Pagot, Aymen Laraoui, Cédric Nim, Rijad Talovic**

---
## Libraries & Configuration

In [23]:
#Libraries import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [24]:
#Data path
data_path = 'Data_2026/' #changer si nécessaire

#Group Parameters
region = "EM"

#Configuration
low_prices_threshold = 0.5
stale_threshold = 0.5 # Proportion max de rendements nuls tolérée (50%)

---
# Part I — Standard Portfolio Allocation

## 1. Data Handling

### 1.1 Data Loading

In [25]:
#Static file Loading and filtering to emerging markets

static_file = pd.read_excel(f'{data_path}Static_2025.xlsx')
emerging_market_data = static_file[static_file['Region'] == region]
#print(emerging_market_data)

#Function to load Data files
def load_data(file_name):
  data = pd.read_excel(f'{data_path}{file_name}')
  data = data.dropna(subset=['ISIN']) # drop error rows
  data = data[data['ISIN'].isin(emerging_market_data['ISIN'])] #Keep emerging markets only
  data = data.set_index('ISIN') #Make ISIN the index
  data = data.drop(columns=['NAME']) #Drop the name column
  data = data.loc[:, data.columns.notna()] #drop columns with na
  data = data.apply(pd.to_numeric, errors='coerce') #Tries to convert every cell to a number and if it can't turn it into NaN

  return data

In [26]:
#Load all the data files

#Return Index
ri_monthly = load_data('DS_RI_T_USD_M_2025.xlsx') # monthly total return index (prices)
ri_yearly  = load_data('DS_RI_T_USD_Y_2025.xlsx') # yearly total return index

#Market Capitalization
mv_monthly = load_data('DS_MV_T_USD_M_2025.xlsx') # monthly market capitalization
mv_yearly  = load_data('DS_MV_T_USD_Y_2025.xlsx') # yearly market capitalization

#Co2 Scope 1 Emissions
co2_scope1 = load_data('DS_CO2_SCOPE_1_Y_2025.xlsx') # yearly CO2 scope 1 emissions (tonnes)
co2_scope1 = co2_scope1.ffill(axis=1) # forward-fill les années manquantes (consigne PDF section 1)

#Revenues
revenues   = load_data('DS_REV_Y_2025.xlsx') # yearly revenues (thousands USD)
revenues   = revenues.ffill(axis=1) # forward-fill les années manquantes (consigne PDF section 1)

#petit aperçu de la data
print(static_file.columns.tolist())
print(emerging_market_data[['NAME', 'Country', 'Region']].head())

['ISIN', 'NAME', 'Country', 'Region']
                                   NAME Country Region
1                                 ALUAR      AR     EM
2                  BANCO BBVA ARGENTINA      AR     EM
3    TERNIUM ARGENTINA SOCIEDAD ANONIMA      AR     EM
117              CHINA YURUN FOOD GROUP      CN     EM
144                            AMBEV ON      BR     EM


In [27]:
#Load the Risk Free Rate

rf_raw = pd.read_excel(f'{data_path}Risk_Free_Rate_2025.xlsx')
rf_raw.columns = ['Date', 'Risk Free Rate']
rf_raw['Date'] = pd.to_datetime(rf_raw['Date'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
rf = rf_raw.set_index('Date')['Risk Free Rate'] / 100

### 1.2 Data Cleaning

In [28]:
#Remove the firms with no data

# Prix bas : Remplacer les valeurs strictement inférieures au seuil par NaN
ri_monthly[ri_monthly < low_prices_threshold] = np.nan

# Boucher les trous : Combler les valeurs manquantes isolées avec le prix précédent
ri_monthly = ri_monthly.ffill(limit=1, axis=1)

In [29]:
# Calculer les rendements mensuels à partir des prix
prices = ri_monthly.copy()

# Calculer les rendements simples (pct_change sur les colonnes = dans le temps)
returns = prices.pct_change(axis=1)

# Gérer les faillites (delistings) : forcer le rendement à -100%, le masque repère : mois t-1 avait un prix ET mois t est vide
mask_delisted = prices.notna().shift(1, axis=1) & prices.isna()
returns[mask_delisted] = -1.0

# Nettoyer les infinis (divisions par zéro)
returns = returns.replace([np.inf, -np.inf], np.nan)

print(f"Shape des returns : {returns.shape}")
print(f"Valeurs infinies restantes : {np.isinf(returns.values).sum()}")
print(returns.head())

Shape des returns : (702, 314)
Valeurs infinies restantes : 0
              1999-12-31 00:00:00  2000-01-31 00:00:00  2000-02-29 00:00:00  \
ISIN                                                                          
ARALUA010258                  NaN             0.176511             0.050002   
ARP125991090                  NaN            -0.051861             0.178923   
ARSIDE010029                  NaN            -0.004877             0.039543   
BMG211591018                  NaN                  NaN                  NaN   
BRABEVACNOR1                  NaN            -0.044175            -0.045539   

              2000-03-31 00:00:00  2000-04-28 00:00:00  2000-05-31 00:00:00  \
ISIN                                                                          
ARALUA010258            -0.023552            -0.016173            -0.016727   
ARP125991090            -0.105072            -0.161842             0.005304   
ARSIDE010029            -0.002113            -0.071188            -0

---
## 2. Minimum-Variance Portfolio Allocation

### 2.1 Investment Set

Pour chaque année d'investissement Y (de 2014 à 2024), on constitue l'univers investissable à partir des données disponibles fin Y-1. Les variables construites sont :

| Variable | Description |
|---|---|
| `emerging_market_data` | Entreprises EM (filtre région) |
| `eligible_isins` | ISINs EM ayant déclaré leurs émissions CO2 fin Y-1 |
| `filtered_isins` | Survivants aux 3 filtres financiers (historique, stale, dernier prix) |
| `filtered_carbon_isin_dict` | Dictionnaire final `{année: [ISINs éligibles]}` pour l'optimisation |

In [30]:
# Pour avoir le set d'investissment possible par année, on doit appliquer les filtres suivants :

filtered_carbon_isin_dict = {}

for year in range(2014, 2025):

    # Fenêtre de temps dynamique de 10 ans
    end_date = returns.columns[returns.columns.astype(str).str.contains(str(year-1))].max()
    end_idx = returns.columns.get_loc(end_date)
    start_idx = end_idx - 119
    window_cols = returns.columns[start_idx:end_idx+1]

    # ÉTAPE 2 : FILTRE CARBONE — ISINs ayant déclaré des émissions CO2 pour Y-1
    prev_year = year - 1
    matching_cols = [col for col in co2_scope1.columns
                     if str(col) == str(prev_year) or (hasattr(col, 'year') and col.year == prev_year)]
    eligible_isins = co2_scope1[matching_cols[0]].dropna().index.tolist() if matching_cols else []

    filtered_isins = []

    # ÉTAPE 3 : FILTRES FINANCIERS
    for isin in eligible_isins:
        if isin not in returns.index:
            continue
        returns_series = returns.loc[isin, window_cols]

        # FILTRE A : min 36 mois valides sur 120
        has_enough_data = returns_series.notna().sum() >= 36

        # FILTRE B : exclure si proportion de rendements nuls > stale_threshold
        has_stale_prices = (returns_series == 0.0).sum() / 120 > stale_threshold

        # FILTRE C : dernier prix valide
        last_return_valid = pd.notna(returns_series.iloc[-1])

        if has_enough_data and not has_stale_prices and last_return_valid:
            filtered_isins.append(isin)

    filtered_carbon_isin_dict[year] = filtered_isins

    print(f"Annee {year} (Fenetre du {window_cols[0].date()} au {window_cols[-1].date()}) :")
    print(f"  -> {len(filtered_isins)} actions conservees sur {len(eligible_isins)} eligibles au depart.")

Annee 2014 (Fenetre du 2004-01-30 au 2013-12-31) :
  -> 254 actions conservees sur 286 eligibles au depart.
Annee 2015 (Fenetre du 2005-01-31 au 2014-12-31) :
  -> 280 actions conservees sur 313 eligibles au depart.
Annee 2016 (Fenetre du 2006-01-31 au 2015-12-31) :
  -> 312 actions conservees sur 349 eligibles au depart.
Annee 2017 (Fenetre du 2007-01-31 au 2016-12-30) :
  -> 346 actions conservees sur 383 eligibles au depart.
Annee 2018 (Fenetre du 2008-01-31 au 2017-12-29) :
  -> 390 actions conservees sur 428 eligibles au depart.
Annee 2019 (Fenetre du 2009-01-30 au 2018-12-31) :
  -> 425 actions conservees sur 467 eligibles au depart.
Annee 2020 (Fenetre du 2010-01-29 au 2019-12-31) :
  -> 472 actions conservees sur 521 eligibles au depart.
Annee 2021 (Fenetre du 2011-01-31 au 2020-12-31) :
  -> 510 actions conservees sur 565 eligibles au depart.
Annee 2022 (Fenetre du 2012-01-31 au 2021-12-31) :
  -> 547 actions conservees sur 603 eligibles au depart.
Annee 2023 (Fenetre du 2013-

### 2.2 Minimum-Variance Portfolio

### 2.3 Comparison with the Value-Weighted Portfolio

---
# Part II — Portfolio Allocation with Carbon Emission Reduction

## 3. Allocation with a 50% Reduction in Carbon Emissions

### 3.1 Carbon Emissions

### 3.2 Long-Only Portfolio with a Carbon Footprint Objective

### 3.3 Tracking Error Minimization

### 3.4 Comparison of Portfolios

---
## 4. Portfolio Allocation with a Net Zero Objective

### 4.1 Net Zero Portfolio

### 4.2 Comparison of Portfolios